In [11]:
import sys
!{sys.executable} -m pip install chromadb
!{sys.executable} -m pip install google-genai chromadb
!{sys.executable} -m pip install google-generativeai


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\shahj\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\shahj\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.3 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/15.6 MB 8.3 MB/s eta 0:00:02
   ----- ---------------------------------- 2.1/15.6 MB 5.1 MB/s eta 0:00:03
   --------- ------------------------------ 3.

  You can safely remove it manually.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\shahj\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
#create collections

import chromadb
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(name="my_collection")

collection.add(
    ids=["id1", "id2"],
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges"
    ]
)

results = collection.query(
    query_texts=["This is a query document about hawaii"], # Chroma will embed this for you
    n_results=2 # how many results to return
)
print(results)

C:\Users\shahj\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:29<00:00, 2.80MiB/s]


{'ids': [['id1', 'id2']], 'embeddings': None, 'documents': [['This is a document about pineapple', 'This is a document about oranges']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[1.0404013395309448, 1.2430801391601562]]}


In [16]:
# See collections

import os

with open("C:/Users/shahj/OneDrive/Documents/Esiee/2e_annee/Stage/Ressources/AI_Engineering/Week5_LLM_RAG_LangChain_OpenAI/Gemini_API_Key.txt", "r") as file:
    api_key = file.read()

os.environ["GEMINI_API_KEY"] = api_key

import chromadb
from chromadb.utils.embedding_functions import GoogleGeminiEmbeddingFunction

client = chromadb.PersistentClient(path="./chroma_db")

gemini_ef = GoogleGeminiEmbeddingFunction(
    #api_key=os.environ.get("GEMINI_API_KEY"),
    model_name="models/gemini-embedding-001"
)

collection = client.get_or_create_collection(
    name="my_gemini_collection",
    embedding_function=gemini_ef
)

print("--- Collections list ---")
print(client.list_collections())

collection = client.get_collection(name="my_gemini_collection")

print("\n--- Stored documents ---")
print(collection.count())

print("\n--- First data overview ---")
print(collection.peek(limit=2))

--- Liste des collections ---
[Collection(name=my_gemini_collection)]

--- Nombre de documents stockés ---
0

--- Aperçu des premières données ---
{'ids': [], 'embeddings': array([], dtype=float64), 'documents': [], 'uris': None, 'included': ['metadatas', 'documents', 'embeddings'], 'data': None, 'metadatas': []}


In [20]:
#Add data

import os
import chromadb
from chromadb.utils.embedding_functions import GoogleGeminiEmbeddingFunction

client = chromadb.Client()

gemini_ef = GoogleGeminiEmbeddingFunction(
    #api_key=os.getenv("GEMINI_API_KEY"),
    model_name="models/gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT" # optimize RAG storage
)
collection = client.get_or_create_collection(name="gemini_collection", embedding_function=gemini_ef)

collection.add(
    ids=["doc_1", "doc_2"],
    documents=["Asymetric crypting uses two keys.", "Hashing can't be undone."],
    metadatas=[{"topic": "crypto", "level": 1}, {"topic": "crypto", "level": 2}]
)

print("--- Collections list ---")
print(client.list_collections())

collection = client.get_collection(name="gemini_collection")

print("\n--- Stored documents ---")
print(collection.count())

print("\n--- First data overview ---")
print(collection.peek(limit=2))

--- Collections list ---
[Collection(name=my_collection), Collection(name=gemini_collection)]

--- Stored documents ---
2

--- First data overview ---
{'ids': ['doc_1', 'doc_2'], 'embeddings': array([[-0.01349459, -0.00331346, -0.0096155 , ..., -0.00353825,
         0.00148931, -0.0144798 ],
       [-0.0095477 , -0.00134425,  0.00494522, ..., -0.00352463,
         0.01288259, -0.00511947]], shape=(2, 3072)), 'documents': ['Asymetric crypting uses two keys.', "hashing can't be undone."], 'uris': None, 'included': ['metadatas', 'documents', 'embeddings'], 'data': None, 'metadatas': [{'topic': 'crypto', 'level': 1}, {'level': 2, 'topic': 'crypto'}]}


In [22]:
#Query and Get
import os
import chromadb
from chromadb.utils.embedding_functions import GoogleGeminiEmbeddingFunction

client = chromadb.Client()

gemini_query_ef = GoogleGeminiEmbeddingFunction(
    #api_key=os.getenv("GEMINI_API_KEY"),
    model_name="models/gemini-embedding-001",
    task_type="RETRIEVAL_QUERY"
)
collection = client.get_collection(name="gemini_collection", embedding_function=gemini_query_ef)

results = collection.query(
    query_texts=["How does crypting work ?"],
    n_results=2
)
print(results["documents"][0][0])


Asymetric crypting uses two keys.
